# NB3 — Plutchik Model v2: Training
**Person 3 runs this notebook AFTER NB1 and NB2 are complete.**

Architecture: DeBERTa-v3-base + **8 separate EmotionAttentionBlocks** (one per Plutchik axis) + per-emotion temperature scaling.

## Setup before running:
1. In Kaggle → `+ Add Data` → search for `plutchik-labels-p1` (from Person 1) → add it
2. In Kaggle → `+ Add Data` → search for `plutchik-labels-p2` (from Person 2) → add it
3. Enable GPU: Settings → Accelerator → T4 x1
4. Run all cells

- Expected runtime: ~4 hours for 12 curriculum epochs
- Output: `/kaggle/working/plutchik_model_v2/` → share as `plutchik-model-v2`


## 0. Install

In [ ]:
%%capture
!pip install -U requests -q
!pip install datasets==2.19.0 -q
!pip install transformers==4.44.0 accelerate scipy scikit-learn -q
print('done')

## 1. Config

In [ ]:
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── Data paths (Kaggle input datasets) ────────────────────────────────────
P1_CSV   = '/kaggle/input/plutchik-labels-p1/plutchik_labeled_p1.csv'
P2_CSV   = '/kaggle/input/plutchik-labels-p2/plutchik_labeled_p2.csv'
SE_CSV   = '/kaggle/input/plutchik-labels-p2/semeval_continuous.csv'
OUT_DIR  = '/kaggle/working/plutchik_model_v2'
os.makedirs(OUT_DIR, exist_ok=True)

# ── Model ──────────────────────────────────────────────────────────────────
BASE_MODEL   = 'microsoft/deberta-v3-base'
EMOTIONS     = ['joy','trust','fear','surprise','sadness','disgust','anger','anticipation']
N_EMOTIONS   = len(EMOTIONS)

# ── Training ───────────────────────────────────────────────────────────────
MAX_LEN     = 128
BATCH_SIZE  = 16
GRAD_ACCUM  = 2          # effective batch = 32
TOTAL_EPOCHS = 12        # 4 per curriculum stage (easy/medium/all)
WARMUP_RATIO = 0.06
BASE_LR     = 1e-5       # encoder LR (lower than before to avoid forgetting)
HEAD_LR     = 2e-5       # attention blocks + heads — lower to prevent NaN on random init
LR_DECAY    = 0.85       # per encoder layer
WEIGHT_DECAY = 0.01

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Config: {TOTAL_EPOCHS} epochs, batch={BATCH_SIZE}x{GRAD_ACCUM}={BATCH_SIZE*GRAD_ACCUM} effective')

## 2. Load & Merge Data

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import load_dataset


def load_go_emotions_baseline():
    """Fallback: remap GoEmotions 27-label to soft 8D Plutchik with label smoothing."""
    GO2PLU = {
        'admiration':'trust','amusement':'joy','anger':'anger','annoyance':'anger',
        'approval':'trust','caring':'trust','confusion':'surprise','curiosity':'anticipation',
        'desire':'anticipation','disappointment':'sadness','disapproval':'disgust',
        'disgust':'disgust','embarrassment':'fear','excitement':'anticipation',
        'fear':'fear','gratitude':'joy','grief':'sadness','joy':'joy','love':'joy',
        'nervousness':'fear','optimism':'anticipation','pride':'joy','realization':'surprise',
        'relief':'joy','remorse':'sadness','sadness':'sadness','surprise':'surprise',
        'neutral':'joy',
    }
    rows = []
    EPS = 0.05  # label smoothing — prevents hard one-hot targets
    go = load_dataset('go_emotions', 'simplified', split='train')
    label_names = go.features['labels'].feature.names
    for row in go:
        vec = {e: EPS for e in EMOTIONS}  # start with smoothed zeros
        if row['labels']:
            for lid in row['labels']:
                lname = label_names[lid]
                plutchik_emo = GO2PLU.get(lname)
                if plutchik_emo:
                    vec[plutchik_emo] = min(1.0, vec[plutchik_emo] + (1.0 - EPS))
        # Normalize so max is 1.0
        maxv = max(vec.values())
        if maxv > 0:
            vec = {k: v/maxv for k,v in vec.items()}
        rows.append({'text': row['text'], 'confidence': maxv, **vec})
    return pd.DataFrame(rows)


# ── Load labeled data ─────────────────────────────────────────────────────
dfs = []

if os.path.exists(P1_CSV):
    df1 = pd.read_csv(P1_CSV)
    dfs.append(df1)
    print(f'Loaded NB1 labels : {len(df1):,} rows')

if os.path.exists(P2_CSV):
    df2 = pd.read_csv(P2_CSV)
    dfs.append(df2)
    print(f'Loaded NB2 labels : {len(df2):,} rows')

if os.path.exists(SE_CSV):
    dse = pd.read_csv(SE_CSV)
    dfs.append(dse)
    print(f'Loaded SemEval    : {len(dse):,} rows')

if not dfs:
    print('WARNING: No labeled CSVs found. Using GoEmotions baseline with label smoothing.')
    print('For best results, add plutchik-labels-p1 and plutchik-labels-p2 as Kaggle datasets.')
    base_df = load_go_emotions_baseline()
    dfs.append(base_df)

full_df = pd.concat(dfs, ignore_index=True).drop_duplicates(subset='text')
# Clip all emotion values to [0, 1]
full_df[EMOTIONS] = full_df[EMOTIONS].clip(0.0, 1.0)
full_df['confidence'] = full_df['confidence'].clip(0.0, 1.0)
print(f'Total unique sentences: {len(full_df):,}')

## 3. Train/Val/Test Split + Curriculum Scoring

In [ ]:
# Split: 80% train, 10% val, 10% test
train_df, temp_df = train_test_split(full_df, test_size=0.2, random_state=SEED)
val_df, test_df   = train_test_split(temp_df, test_size=0.5, random_state=SEED)
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')


def compute_difficulty(df):
    """Higher = harder. Based on multi-emotion ambiguity and low confidence."""
    labels = df[EMOTIONS].values.astype(np.float32)
    normed = labels / (labels.sum(axis=1, keepdims=True) + 1e-8)
    entropy = -np.sum(normed * np.log(normed + 1e-8), axis=1) / np.log(N_EMOTIONS)
    n_active = (labels > 0.2).sum(axis=1) / N_EMOTIONS
    conf_penalty = 1.0 - df['confidence'].values
    return 0.4 * entropy + 0.4 * n_active + 0.2 * conf_penalty


train_df = train_df.copy().reset_index(drop=True)
train_df['difficulty'] = compute_difficulty(train_df)
difficulty_order = np.argsort(train_df['difficulty'].values)  # easiest first

# Curriculum stages: easy(33%) → medium(66%) → all
n = len(difficulty_order)
CURRICULUM = [
    ('easy',   difficulty_order[:n//3],   4),  # 4 epochs on easiest 33%
    ('medium', difficulty_order[:2*n//3], 4),  # 4 epochs on easiest 66%
    ('all',    difficulty_order,          4),  # 4 epochs on all
]
print(f'Curriculum: {[(s, len(idx), e) for s,idx,e in CURRICULUM]}')
print(f'Total epochs: {sum(e for _,_,e in CURRICULUM)}')

## 4. Dataset & DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader, Subset
from transformers import AutoTokenizer

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)


class PlutchikDataset(Dataset):
    def __init__(self, df, max_len=MAX_LEN):
        self.texts  = df['text'].tolist()
        self.labels = df[EMOTIONS].values.astype(np.float32)
        self.conf   = df['confidence'].values.astype(np.float32)
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'token_type_ids': enc.get('token_type_ids', torch.zeros(self.max_len, dtype=torch.long)).squeeze(0) if 'token_type_ids' in enc else torch.zeros(self.max_len, dtype=torch.long),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.float32),
            'confidence':     torch.tensor(self.conf[idx], dtype=torch.float32),
        }


train_dataset = PlutchikDataset(train_df)
val_dataset   = PlutchikDataset(val_df)
test_dataset  = PlutchikDataset(test_df)

val_loader  = DataLoader(val_dataset,  batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)
print('Datasets ready')

## 5. Model Architecture — 8 Separate EmotionAttentionBlocks

In [ ]:
from transformers import AutoModel


class EmotionAttentionBlock(nn.Module):
    """
    Dedicated attention block for ONE Plutchik emotion.
    Each of the 8 emotions has its own:
      - Learnable query vector
      - Key projection (K)
      - Value projection (V)
      - Output projection
    This lets each emotion learn to look at different
    token positions independently.
    """
    def __init__(self, hidden_size, n_heads=4, out_dim=128):
        super().__init__()
        assert hidden_size % n_heads == 0
        self.n_heads  = n_heads
        self.head_dim = hidden_size // n_heads
        self.scale    = self.head_dim ** -0.5

        # Each emotion: dedicated learnable query
        self.query  = nn.Parameter(torch.randn(1, 1, hidden_size) * 0.02)

        # Each emotion: dedicated K/V projections (not shared with other emotions)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)

        self.out_proj = nn.Linear(hidden_size, out_dim)
        self.norm     = nn.LayerNorm(out_dim)
        self.dropout  = nn.Dropout(0.1)

    def forward(self, hidden_states, attention_mask):
        B, L, H = hidden_states.shape
        nh, hd  = self.n_heads, self.head_dim

        q = self.query.expand(B, -1, -1)       # (B, 1, H)
        k = self.k_proj(hidden_states)          # (B, L, H)
        v = self.v_proj(hidden_states)          # (B, L, H)

        # Reshape to multi-head
        q = q.view(B, 1, nh, hd).transpose(1, 2)    # (B, nh, 1, hd)
        k = k.view(B, L, nh, hd).transpose(1, 2)    # (B, nh, L, hd)
        v = v.view(B, L, nh, hd).transpose(1, 2)    # (B, nh, L, hd)

        attn = (q @ k.transpose(-2, -1)) * self.scale  # (B, nh, 1, L)

        # Mask padding positions
        if attention_mask is not None:
            pad = (attention_mask == 0).unsqueeze(1).unsqueeze(2)  # (B,1,1,L)
            attn = attn.masked_fill(pad, float('-inf'))

        attn = self.dropout(torch.softmax(attn, dim=-1))

        out = (attn @ v).squeeze(2).reshape(B, H)   # (B, H)
        out = self.norm(self.out_proj(out))          # (B, out_dim)
        return out


class PlutchikModelV2(nn.Module):
    """
    DeBERTa-v3-base + 8 separate EmotionAttentionBlocks.
    Per-emotion temperature scaling fixes score compression.
    """
    def __init__(self, base_model_name=BASE_MODEL, n_emotions=N_EMOTIONS,
                 out_dim=128, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_name)
        H = self.encoder.config.hidden_size

        # 8 SEPARATE blocks — one per Plutchik axis
        self.emotion_blocks = nn.ModuleList([
            EmotionAttentionBlock(H, n_heads=4, out_dim=out_dim)
            for _ in range(n_emotions)
        ])

        # Per-emotion regression head (shared structure, separate weights)
        self.emotion_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(out_dim, 32),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(32, 1),
            ) for _ in range(n_emotions)
        ])

        # Per-emotion temperature — fixes score compression.
        # Init at 1.0 (safe) — 2.0 caused NaN with random head weights.
        self.temperature = nn.Parameter(torch.full((n_emotions,), 1.0))

        # Small weight init on heads — prevents large logits at start (NaN cause)
        for head in self.emotion_heads:
            for layer in head:
                if hasattr(layer, 'weight'):
                    nn.init.xavier_uniform_(layer.weight, gain=0.1)
                if hasattr(layer, 'bias') and layer.bias is not None:
                    nn.init.zeros_(layer.bias)

        # Confidence head (CLS token)
        self.conf_head = nn.Sequential(
            nn.Linear(H, 128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, 1), nn.Sigmoid()
        )

        # Auxiliary classification head — predicts dominant emotion from CLS.
        # Discrete supervision on top of regression; helps weak emotions and
        # the dominant-accuracy plateau (~64%).
        self.cls_head = nn.Sequential(
            nn.Linear(H, 128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, n_emotions),
        )

        self.drop = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        enc_out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        hidden = self.drop(enc_out.last_hidden_state)  # (B, L, H)
        cls    = hidden[:, 0]                          # (B, H)

        # Each emotion block independently attends
        logits = []
        for block, head in zip(self.emotion_blocks, self.emotion_heads):
            feat  = block(hidden, attention_mask)   # (B, out_dim)
            logit = head(feat)                      # (B, 1)
            logits.append(logit)

        logits = torch.cat(logits, dim=-1)           # (B, 8)

        # Temperature scaling — prevents near-zero collapse
        temp   = torch.clamp(self.temperature, min=0.5, max=5.0)
        scores = torch.sigmoid(logits * temp)        # (B, 8)

        conf       = self.conf_head(cls)             # (B, 1)
        cls_logits = self.cls_head(cls)               # (B, n_emotions)
        return scores, conf.squeeze(-1), cls_logits


print('Architecture defined')
# Count params
_m = PlutchikModelV2()
total = sum(p.numel() for p in _m.parameters())
new   = sum(p.numel() for p in list(_m.emotion_blocks.parameters()) +
            list(_m.emotion_heads.parameters()) +
            list(_m.conf_head.parameters()) + [_m.temperature])
print(f'Total params : {total/1e6:.1f}M')
print(f'New params   : {new/1e6:.1f}M  (emotion blocks + heads)')
del _m

## 6. Loss Functions

In [ ]:
# Per-emotion weights — boost fear/surprise (chronic underperformers in baseline)
# Order: joy, trust, fear, surprise, sadness, disgust, anger, anticipation
EMOTION_WEIGHTS = torch.tensor([1.0, 1.0, 1.3, 1.3, 1.1, 1.1, 1.0, 1.0]).to(DEVICE)


def cosine_loss(pred, target):
    return 1.0 - F.cosine_similarity(pred, target, dim=1).mean()


def rank_loss(pred, target, margin=0.05):
    """Penalise ordering violations across all 28 emotion pairs."""
    loss, count = 0.0, 0
    for i in range(N_EMOTIONS):
        for j in range(i+1, N_EMOTIONS):
            diff_t = target[:, i] - target[:, j]
            diff_p = pred[:, i]   - pred[:, j]
            sign   = torch.sign(diff_t)
            mask   = diff_t.abs() > margin
            if mask.any():
                loss  += F.relu(margin - sign * diff_p)[mask].mean()
                count += 1
    return loss / max(count, 1)


def weighted_smoothed_mse(pred, target, eps=0.03):
    """
    Per-emotion weighted MSE with mild label smoothing (0.0 -> 0.03, 1.0 -> 0.97).
    Smoothing prevents punishment of small positive preds on 'zero' axes.
    Weighting upweights fear/surprise so dominant emotions don't drown them out.
    """
    smoothed = target * (1 - 2*eps) + eps
    per_dim  = (pred - smoothed).pow(2).mean(dim=0)   # (N_EMOTIONS,)
    return (EMOTION_WEIGHTS * per_dim).mean()


def aux_cls_loss(cls_logits, target):
    """
    Auxiliary classification: predict the dominant emotion (argmax of soft target).
    Adds discrete supervision on top of the regression objective.
    """
    dom = target.argmax(dim=-1)
    return F.cross_entropy(cls_logits, dom)


def combined_loss(scores, labels, conf, target_conf, cls_logits=None):
    # Rank loss reweighted UP — Spearman is the headline metric and was lagging.
    emotion_loss = (
        0.35 * cosine_loss(scores, labels)
      + 0.50 * rank_loss(scores, labels)
      + 0.15 * weighted_smoothed_mse(scores, labels)
    )
    conf_loss = F.mse_loss(conf, target_conf)
    total = emotion_loss + 0.05 * conf_loss
    if cls_logits is not None:
        total = total + 0.2 * aux_cls_loss(cls_logits, labels)
    return total


print('Loss functions ready')

## 7. Optimizer — Layer-wise LR Decay

In [ ]:
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup


def build_optimizer(model, base_lr=BASE_LR, head_lr=HEAD_LR, decay=LR_DECAY):
    """
    - Encoder layers get base_lr * decay^(depth from top)
    - EmotionAttentionBlocks + heads get head_lr (train from scratch)
    - Temperature parameters get head_lr
    """
    groups = []
    seen   = set()

    def add(params, lr):
        new = [p for p in params if id(p) not in seen and p.requires_grad]
        if new:
            groups.append({'params': new, 'lr': lr, 'weight_decay': WEIGHT_DECAY})
            seen.update(id(p) for p in new)

    # New heads (high LR — training from scratch)
    add(list(model.emotion_blocks.parameters()) +
        list(model.emotion_heads.parameters()) +
        list(model.conf_head.parameters()) +
        list(model.cls_head.parameters()) +
        [model.temperature], head_lr)

    # Encoder layers (layer-wise decay)
    enc_layers = model.encoder.encoder.layer
    n = len(enc_layers)
    for k, layer in enumerate(reversed(enc_layers)):
        add(list(layer.parameters()), base_lr * (decay ** k))

    # Embeddings (lowest LR)
    add(list(model.encoder.embeddings.parameters()), base_lr * (decay ** n))

    return AdamW(groups)


print('Optimizer builder ready')

## 8. Evaluation

In [ ]:
from scipy.stats import spearmanr, pearsonr


def evaluate(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            ids   = batch['input_ids'].to(DEVICE)
            mask  = batch['attention_mask'].to(DEVICE)
            ttids = batch['token_type_ids'].to(DEVICE)
            scores, _, _ = model(ids, mask, ttids)  # pure FP32, no autocast
            preds.append(scores.float().cpu().numpy())
            targets.append(batch['labels'].numpy())

    preds   = np.concatenate(preds)
    targets = np.concatenate(targets)

    spearmans, pearsons = [], []
    per_emo = {}
    for i, e in enumerate(EMOTIONS):
        if preds[:, i].std() > 1e-6 and targets[:, i].std() > 1e-6:
            rho = spearmanr(preds[:, i], targets[:, i])[0]
            prs = pearsonr(preds[:, i], targets[:, i])[0]
        else:
            rho, prs = 0.0, 0.0
        per_emo[e] = {'spearman': rho, 'pearson': prs}
        spearmans.append(rho); pearsons.append(prs)

    # Vector cosine sim (per sample)
    pn = preds   / (np.linalg.norm(preds,   axis=1, keepdims=True) + 1e-8)
    tn = targets / (np.linalg.norm(targets, axis=1, keepdims=True) + 1e-8)
    cosine = float((pn * tn).sum(axis=1).mean())

    # Dominant emotion accuracy
    dom_acc = float((preds.argmax(1) == targets.argmax(1)).mean())

    return {
        'mean_spearman': float(np.mean(spearmans)),
        'mean_pearson':  float(np.mean(pearsons)),
        'vector_cosine': cosine,
        'dom_acc':       dom_acc,
        'per_emotion':   per_emo,
        'preds':         preds,
        'targets':       targets,
    }


def print_metrics(m, stage=''):
    tag = f' ({stage})' if stage else ''
    print(f'\n── Metrics{tag} ────────────────────────')
    print(f'  Mean Spearman      : {m["mean_spearman"]:.4f}')
    print(f'  Vector Cosine      : {m["vector_cosine"]:.4f}')
    print(f'  Mean Pearson       : {m["mean_pearson"]:.4f}')
    print(f'  Dominant Emo Acc   : {m["dom_acc"]:.4f}')
    print('  Per-emotion Spearman:')
    for e, v in m['per_emotion'].items():
        bar = chr(9608) * int(max(0, v['spearman']) * 20)
        print(f'    {e:<15}: {v["spearman"]:+.4f}  {bar}')


print('Evaluation ready')

## 9. Training Loop with Curriculum Learning

In [ ]:
import math, gc

# Force FP32 throughout — DeBERTa BF16 causes NaN in disentangled attention
AMP_DTYPE = torch.float32
USE_AMP   = False
print(f'Training dtype: float32 (AMP disabled)')

# ── Init model fully in float32 BEFORE optimizer ───────────────────────────
gc.collect()
torch.cuda.empty_cache()
model = PlutchikModelV2().to(DEVICE).float()  # .float() ensures ALL params are FP32
print(f'Encoder dtype : {next(model.encoder.parameters()).dtype}')
print(f'Block 0 dtype : {next(model.emotion_blocks[0].parameters()).dtype}')

# ── Build optimizer AFTER model is in FP32 ────────────────────────────────
total_steps = sum(
    math.ceil(len(idx) / BATCH_SIZE) * epochs * GRAD_ACCUM
    for _, idx, epochs in CURRICULUM
)
optimizer  = build_optimizer(model)
scheduler  = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(WARMUP_RATIO * total_steps),
    num_training_steps=total_steps,
)

best_spearman   = -1.0
best_val_loss   = float('inf')
patience        = 3   # epochs without val-loss improvement before we stop
patience_ctr    = 0
early_stop      = False
history     = []
global_step = 0


def compute_val_loss(model, loader):
    """Mirrors the training loss so early stopping reflects what we optimise."""
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            ids   = batch['input_ids'].to(DEVICE)
            mask  = batch['attention_mask'].to(DEVICE)
            ttids = batch['token_type_ids'].to(DEVICE)
            lab   = batch['labels'].to(DEVICE)
            tcf   = batch['confidence'].to(DEVICE)
            scores, conf, cls_logits = model(ids, mask, ttids)
            l = combined_loss(scores, lab, conf, tcf, cls_logits)
            total += l.item()
            n += 1
    return total / max(n, 1)


for stage_name, stage_idx, stage_epochs in CURRICULUM:
    stage_subset = Subset(train_dataset, stage_idx.tolist())
    print(f'\n{"="*60}')
    print(f'Curriculum stage: {stage_name}  ({len(stage_subset):,} samples, {stage_epochs} epochs)')
    print(f'{'='*60}')

    for epoch in range(stage_epochs):
        stage_loader = DataLoader(stage_subset, batch_size=BATCH_SIZE,
                                  shuffle=True, num_workers=2, pin_memory=True)
        model.train()
        total_loss = cos_l = rank_l = mse_l = 0.0
        optimizer.zero_grad()

        for step, batch in enumerate(stage_loader):
            ids    = batch['input_ids'].to(DEVICE)
            mask   = batch['attention_mask'].to(DEVICE)
            ttids  = batch['token_type_ids'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            tconf  = batch['confidence'].to(DEVICE)

            # No autocast — pure FP32
            scores, conf, cls_logits = model(ids, mask, ttids)
            cl   = cosine_loss(scores, labels)
            rl   = rank_loss(scores, labels)
            ml   = weighted_smoothed_mse(scores, labels)
            al   = aux_cls_loss(cls_logits, labels)
            loss = (0.35*cl + 0.50*rl + 0.15*ml
                    + 0.05*F.mse_loss(conf, tconf)
                    + 0.20*al) / GRAD_ACCUM

            # NaN guard — skip bad batches instead of corrupting weights
            if torch.isnan(loss):
                optimizer.zero_grad()
                continue

            loss.backward()

            total_loss += loss.item() * GRAD_ACCUM
            cos_l  += cl.item()
            rank_l += rl.item()
            mse_l  += ml.item()

            if (step + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

        n_steps = len(stage_loader)
        avg_loss = total_loss / n_steps
        print(f'  Epoch {epoch+1}/{stage_epochs} | Loss: {avg_loss:.4f}  '
              f'[cos={cos_l/n_steps:.3f} rank={rank_l/n_steps:.3f} mse={mse_l/n_steps:.3f}]')

        # Val eval every epoch
        val_m    = evaluate(model, val_loader)
        val_loss = compute_val_loss(model, val_loader)
        print_metrics(val_m, f'{stage_name}/epoch{epoch+1}')
        print(f'  Val loss: {val_loss:.4f}  (best: {best_val_loss:.4f}, patience: {patience_ctr}/{patience})')

        h_row = {'stage': stage_name, 'epoch': epoch+1, 'val_loss': val_loss,
                 **{k: v for k,v in val_m.items() if k != 'per_emotion' and k not in ('preds','targets')}}
        history.append(h_row)

        # Save best by Spearman (the headline metric we report on)
        if val_m['mean_spearman'] > best_spearman:
            best_spearman = val_m['mean_spearman']
            torch.save(model.state_dict(), os.path.join(OUT_DIR, 'best_model.pt'))
            print(f'  New best saved (Spearman: {best_spearman:.4f})')

        # Early stopping driven by val LOSS — independent signal from Spearman.
        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            patience_ctr  = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f'  Early stopping: val loss did not improve for {patience} epochs')
                early_stop = True
                break

    if early_stop:
        break

print(f'\nTraining complete | Best val Spearman: {best_spearman:.4f} | Best val loss: {best_val_loss:.4f}')

## 10. Final Test Evaluation

In [ ]:
model.load_state_dict(torch.load(os.path.join(OUT_DIR, 'best_model.pt')))
test_m = evaluate(model, test_loader)
print_metrics(test_m, 'FINAL TEST')

## 11. Save Model, Tokenizer & Metadata

In [ ]:
import json as _json
import pandas as pd

# Tokenizer
tokenizer.save_pretrained(OUT_DIR)

# Model weights already saved as best_model.pt
# Save config
cfg = {
    'base_model': BASE_MODEL,
    'emotions': EMOTIONS,
    'max_len': MAX_LEN,
    'architecture': 'DeBERTa-v3-base + 8x EmotionAttentionBlock + per-emotion temperature',
    'best_val_spearman': best_spearman,
    'test_spearman': test_m['mean_spearman'],
    'test_cosine': test_m['vector_cosine'],
    'test_dom_acc': test_m['dom_acc'],
}
with open(os.path.join(OUT_DIR, 'model_config.json'), 'w') as f:
    _json.dump(cfg, f, indent=2)

pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, 'training_history.csv'), index=False)

print(f'Saved to: {OUT_DIR}')
for fn in os.listdir(OUT_DIR):
    sz = os.path.getsize(os.path.join(OUT_DIR, fn)) / 1e6
    print(f'  {fn:<45} ({sz:.1f} MB)')
print()
print('Share this output as Kaggle dataset: plutchik-model-v2')
print('Person 4 adds it as input to NB4.')

## 12. Auto-Zip Outputs (Kaggle Download)
Kaggle sessions are ephemeral. The zip below appears in `/kaggle/working/` and is downloadable from the session's **Output** tab even if the kernel times out.

In [ ]:
import shutil
from IPython.display import FileLink

zip_base = '/kaggle/working/plutchik_model_v2_outputs'
zip_path = shutil.make_archive(zip_base, 'zip', OUT_DIR)
print(f'Zipped outputs: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)')
FileLink(zip_path)